In [1]:
import pandas as pd
import readability
import os
import numpy as np
import random
import torch
from bert_score import BERTScorer
import re
import pickle
import nltk
from nltk.translate import meteor
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import math

In [2]:
from transformers import logging
logging.set_verbosity_error() # make sure only important transformers logging output is visible

In [3]:
INPUT_DIR = 'processedData'

In [4]:
# Only need to run once.
# nltk.download('all')

In [5]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [6]:
prompt_df = pd.read_csv('../promptDataPreparation/promptDataPreparation.csv')

In [7]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s/]', '', text)
    # tokenize
    tokens = word_tokenize(text.lower())
    # remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    # join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)
    return processed_text

In [8]:
dataset_readabilities = {}
# Get readability of full datasets. 
for folder in os.listdir('../getText/datasetsPrep'):
    if os.path.isdir(f'../getText/datasetsPrep/{folder}'):
        for file in os.listdir(f'../getText/datasetsPrep/{folder}'):
            if file.endswith('.csv'):
                temp_df = pd.read_csv(f'../getText/datasetsPrep/{folder}/{file}')
                text_series = temp_df['text'].fillna('').astype(str).str.replace('.', '.\n', regex=False)
                long_text = '\n'.join(text_series)
                dataset_readabilities[file] = readability.getmeasures(long_text, lang='en')['readability grades']['SMOGIndex']

dataset_readabilities

{'yahoo.csv': 8.457860209518776,
 'banking77.csv': 7.7227892552376485,
 'huffPostNews.csv': 9.800422735858234,
 'clinc150.csv': 7.115884635058898,
 'atis.csv': 9.558259963681241,
 'medicalAbstracts.csv': 14.371525966195462,
 'dementiaAudio.csv': 5.381402423935505,
 'syntheticCareHomeNurseNotes.csv': 10.337798278917836,
 'clinicalDialogueSummarizations.csv': 10.314872711433278,
 'simSUM.csv': 8.856790533456255}

In [9]:
dataset_readabilities['yahoo.csv']

8.457860209518776

In [10]:
average_scores_dict = []
# Make BERT scorer.
scorer = BERTScorer(model_type="bert-base-uncased")
for dataset in os.listdir(f"./{INPUT_DIR}"):
    if dataset.endswith('.csv'):
        temp_df = pd.read_csv(f"./{INPUT_DIR}/{dataset}")
        for topic_model in ['MATAVE', 'LDA']:

            topic_model_df = temp_df[temp_df['topic_model'] == topic_model]
            # --------- Readability (number linked to grade of reading level)s ---------
            references = []
            candidates = []
            meteors = []
            abs_sentiment_diffs = []
            abs_subjectivity_diffs = []
            # Make readability metric.
            reference_readability = dataset_readabilities[dataset]
            text_series = topic_model_df['report'].fillna('').astype(str).str.replace('.', '.\n', regex=False)
            long_text = '\n'.join(text_series)
            candidate_readability = readability.getmeasures(long_text, lang='en')['readability grades']['SMOGIndex']
            abs_readability_diff = (math.sqrt((reference_readability - candidate_readability) ** 2))
            for temp_prompt, temp_generation in zip(topic_model_df['example_text_in_prompt'], topic_model_df['report']):
                # Make candidates and references without punctuation for metrics (BERTScore penalizes punctuation, we should take this into account - https://aclanthology.org/2023.findings-acl.381.pdf).
                reference = re.sub(r'[^\w\s/]', '', temp_prompt)
                candidate = re.sub(r'[^\w\s/]', '', temp_generation)
                references.append(reference)
                candidates.append(candidate)
                # Make METEOR
                meteors.append(meteor([word_tokenize(candidate)], word_tokenize(reference)))
                # Make sentiment and subjectivity absolute differences.
                reference_blob = TextBlob(preprocess_text(temp_prompt)).sentences[0].sentiment
                candidate_blob = TextBlob(preprocess_text(temp_generation)).sentences[0].sentiment
                abs_sentiment_diffs.append(math.sqrt((reference_blob.polarity - candidate_blob.polarity) ** 2))
                abs_subjectivity_diffs.append(math.sqrt((reference_blob.subjectivity - candidate_blob.subjectivity) ** 2))
            # Make BERTScore
            _, _, F1 = scorer.score(candidates, references)

            average_scores_dict.append({
                'topic_model': topic_model, 
                'dataset': dataset, 
                'number_of_notes': len(topic_model_df),
                'abs_readability_diff': abs_readability_diff,
                # save the F1 values, as these are recommended for use by the BERTScore authors - http://arxiv.org/abs/1904.09675
                'bertscore': float(F1.mean()),
                'meteor': sum(meteors) / len(meteors),
                'abs_sentiment_diff': sum(abs_sentiment_diffs) / len(abs_sentiment_diffs),
                'abs_subjectivity_diff': sum(abs_subjectivity_diffs) / len(abs_subjectivity_diffs)})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
# min-max normalize every score
resultant_df = pd.DataFrame(average_scores_dict)
scaled_df = resultant_df.copy()
for column in scaled_df.columns:
    if 'float' in str(scaled_df[column].dtypes):
        scaled_df[column] = (scaled_df[column] - scaled_df[column].min()) / (scaled_df[column].max() - scaled_df[column].min())

In [12]:
resultant_df

,topic_model,dataset,number_of_notes,abs_readability_diff,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff
0,MATAVE,yahoo.csv,726,8.574586,0.453343,0.049080,0.221855,0.309341
1,LDA,yahoo.csv,1925,5.729537,0.456134,0.064448,0.202557,0.231346
2,MATAVE,banking77.csv,760,5.582940,0.503397,0.065937,0.122344,0.258497
3,LDA,banking77.csv,1020,3.438334,0.448749,0.052820,0.200728,0.283857
4,MATAVE,medicalAbstracts.csv,348,7.461355,0.551251,0.242330,0.090604,0.107558
5,LDA,medicalAbstracts.csv,2717,4.716471,0.463367,0.120832,0.129180,0.197455
6,MATAVE,dementiaAudio.csv,1163,7.317842,0.525804,0.146150,0.173821,0.165774
7,LDA,dementiaAudio.csv,1394,5.025586,0.559827,0.190824,0.142839,0.113190
8,MATAVE,huffPostNews.csv,487,4.666307,0.409563,0.032811,0.142061,0.386367
9,LDA,huffPostNews.csv,4512,4.980090,0.372861,0.017203,0.228994,0.299697


In [13]:
scaled_df_all_same_direction = scaled_df.copy()
for column in scaled_df_all_same_direction.columns:
    if 'diff' in column:
        # flip score to make lower results better
        inverse_data = []
        for item in scaled_df_all_same_direction[column]:
            inverse_data.append(1 - item)
        scaled_df_all_same_direction[column] = inverse_data

In [14]:
scaled_df

,topic_model,dataset,number_of_notes,abs_readability_diff,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff
0,MATAVE,yahoo.csv,726,0.862270,0.426361,0.141596,0.952977,0.727648
1,LDA,yahoo.csv,1925,0.430550,0.441144,0.209859,0.825871,0.451864
2,MATAVE,banking77.csv,760,0.408305,0.691522,0.216473,0.297532,0.547869
3,LDA,banking77.csv,1020,0.082874,0.402022,0.158209,0.813823,0.637538
4,MATAVE,medicalAbstracts.csv,348,0.693343,0.945033,1.000000,0.088473,0.014168
5,LDA,medicalAbstracts.csv,2717,0.276823,0.479461,0.460315,0.342557,0.332030
6,MATAVE,dementiaAudio.csv,1163,0.671566,0.810228,0.572775,0.636595,0.220013
7,LDA,dementiaAudio.csv,1394,0.323730,0.990466,0.771215,0.432527,0.034081
8,MATAVE,huffPostNews.csv,487,0.269211,0.194432,0.069327,0.427402,1.000000
9,LDA,huffPostNews.csv,4512,0.316826,0.000000,0.000000,1.000000,0.693547


In [15]:
scaled_df_all_same_direction

,topic_model,dataset,number_of_notes,abs_readability_diff,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff
0,MATAVE,yahoo.csv,726,0.137730,0.426361,0.141596,0.047023,0.272352
1,LDA,yahoo.csv,1925,0.569450,0.441144,0.209859,0.174129,0.548136
2,MATAVE,banking77.csv,760,0.591695,0.691522,0.216473,0.702468,0.452131
3,LDA,banking77.csv,1020,0.917126,0.402022,0.158209,0.186177,0.362462
4,MATAVE,medicalAbstracts.csv,348,0.306657,0.945033,1.000000,0.911527,0.985832
5,LDA,medicalAbstracts.csv,2717,0.723177,0.479461,0.460315,0.657443,0.667970
6,MATAVE,dementiaAudio.csv,1163,0.328434,0.810228,0.572775,0.363405,0.779987
7,LDA,dementiaAudio.csv,1394,0.676270,0.990466,0.771215,0.567473,0.965919
8,MATAVE,huffPostNews.csv,487,0.730789,0.194432,0.069327,0.572598,0.000000
9,LDA,huffPostNews.csv,4512,0.683174,0.000000,0.000000,0.000000,0.306453


In [16]:
scaled_df_all_same_direction['averages'] = scaled_df_all_same_direction[['abs_readability_diff', 'bertscore', 'meteor', 'abs_sentiment_diff', 'abs_subjectivity_diff']].mean(axis=1)

In [17]:
scaled_df_all_same_direction.groupby('topic_model').mean(numeric_only=True)

,number_of_notes,abs_readability_diff,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff,averages
topic_model,,,,,,,
LDA,1976.3,0.599639,0.560841,0.350566,0.453129,0.547952,0.502425
MATAVE,863.5,0.461850,0.646106,0.363691,0.562076,0.534133,0.513571
